#### Setup

In [ ]:
%rebootdevice

In [ ]:
%disconnect

#### Connect board

In [6]:
%serialconnect --port /dev/cu.usbserial-02215408
# /dev/cu.usbserial-02215408

Connecting to --port=/dev/cu.usbserial-02215408 --baud=115200 
Ready.


#### Setup watch

In [ ]:
# Board bring-up with the hal/ drivers (deploy hal/ + finder/ first: tools/deploy.py --noapp).
# The old axp202c/st7789 C-module imports do not exist on stock MicroPython v1.29.
import time
from hal.board import Board
from hal.bma423 import BMA423

b = Board()
pmu = b.pmu                  # AXP202: LDO2 on (panel + backlight), DCDC3 never touched
tft = b.display              # ST7789 on SPI(1), 26.67 MHz, MADCTL 0xC0 + row offset 80
tft.brightness(0.6)
tft.fill(0)
imu = b.imu                  # BMA423 on the shared I2C0: +-4 g, 100 Hz, FIFO on
print("BMA423 chip id 0x%02x, range +-%d g, %d Hz" % (imu.chip_id(), imu.range_g, imu.odr))

#### Why not double-integrate the accelerometer?

The first version of this notebook integrated `get_xyz()` twice to get a position, and it
ran away within seconds. That is not a bug you can tune away:

* **Bias grows quadratically.** A residual offset `b` becomes a position error of `0.5·b·t²`.
  The BMA423's zero-g offset is tens of mg (80 mg typical, plus ~1 mg/K as the watch warms on
  the wrist): 40 mg is already ~20 m after 10 s and ~700 m after 60 s.
* **Gravity leaks in.** Removing 1 g needs the exact tilt; a 1° tilt error leaves 17 mg,
  and the forearm rotates on every step. Integration was also done in the sensor frame,
  which turns with the wrist.
* **No gyro, no compass.** Yaw is unobservable with an accelerometer alone, so even a perfect
  displacement could not be turned into a direction. A Kalman filter cannot recover what the
  sensor does not measure; ZUPT only resets velocity while truly still.
* The old code also had the units wrong (`get_xyz()` already returns g, the ×981/1000 made
  everything ~10× too small) and never removed gravity.

So the game uses what the accelerometer *can* tell reliably: **steps × stride** for distance
walked (a few % error, bounded), **activity/stillness** to know who is moving, and
**tilt / face-up** for gestures and wrist-raise. The Kalman filter goes on the RSSI range
instead. Full analysis and numbers: [docs/estimation/imu-drift.md](../docs/estimation/imu-drift.md)
(reproduce with `python3 tools/drift_demo.py --seeds 5`).

#### Movement: steps, stillness, tilt

In [ ]:
# Steps, stillness and tilt with finder.motion.MotionTracker (fed from the BMA423 FIFO in g).
from finder.motion import MotionTracker

ACT = ("?", "still", "walk", "run")
mt = MotionTracker(rate_hz=50, z_sign=imu.z_sign)   # V1 mounts the chip upside down (hal/pins.py)
imu.fifo_flush()
t_end = time.ticks_add(time.ticks_ms(), 30000)   # 30 s: walk around, stop, tilt the wrist
t_pr = 0
while time.ticks_diff(t_end, time.ticks_ms()) > 0:
    now = time.ticks_ms()
    n = imu.fifo_read_mg()                        # x, y, z milli-g interleaved in imu.fifo_mg
    a = imu.fifo_mg
    for i in range(0, n - 1, 2):                  # 100 Hz -> 50 Hz (pairs averaged), mg -> g
        j = 3 * i
        t = time.ticks_add(now, (i - n + 1) * 10)
        mt.add_sample(t, (a[j] + a[j + 3]) / 2000, (a[j + 1] + a[j + 4]) / 2000,
                      (a[j + 2] + a[j + 5]) / 2000)
    if time.ticks_diff(now, t_pr) >= 500:
        t_pr = now
        print("steps %3d  %4.2f Hz  %-5s  still %d (sd %.3f g)  tilt %5.1f deg  pitch %6.1f  roll %6.1f  face_up %d  walked %.1f m" % (
            mt.steps, mt.step_rate_hz, ACT[mt.activity], mt.is_still, mt.mag_sd_g,
            mt.tilt_deg, mt.pitch_deg, mt.roll_deg, mt.face_up, mt.dist_m))
    time.sleep_ms(40)

#### Drift demo (optional)

In [ ]:
# Optional: see the drift yourself. Leave the watch still on the table and integrate with
# correct units and gravity removed (the first second's mean). It still runs away.
G = 9.80665
imu.fifo_flush(); time.sleep_ms(1000)
n = imu.fifo_read_mg(); a = imu.fifo_mg
g0 = [0.0, 0.0, 0.0]                              # mean mg over 1 s = gravity + bias
for i in range(n):
    for k in range(3):
        g0[k] += a[3 * i + k] / n
v = [0.0, 0.0, 0.0]; x = [0.0, 0.0, 0.0]
for sec in range(10):
    time.sleep_ms(1000)
    n = imu.fifo_read_mg(); a = imu.fifo_mg
    for i in range(n):
        for k in range(3):
            acc = (a[3 * i + k] - g0[k]) * G / 1000
            v[k] += acc * 0.01
            x[k] += v[k] * 0.01
    print("t=%2d s  |x| = %.2f m  (a watch lying still)" % (sec + 1, (x[0]**2 + x[1]**2 + x[2]**2) ** 0.5))